# 02 — MRI Preprocessing + Augmentation

1. Loads Baby Open Brains BIDS dataset (10 T1w/T2w subjects)
2. Extracts axial/coronal/sagittal slices → `datasets/processed/mri/` (10 real `.npz`)
3. **Runs `balance_mri.py` to generate 10,000 synthetic augmented samples** → `datasets/mri/augmented/`

**Run order: 01 → 02 → 03 → 04 → 05 → 06**

In [ ]:
# CELL 1: Clone / pull repo + configure git identity
import os
if not os.path.exists('/content/earlyMind'):
    !git clone https://github.com/Rickykapoor/earlyMind.git /content/earlyMind
%cd /content/earlyMind
!git pull origin main

# Configure git identity (required for any commit)
!git config user.email "rickykapoor@example.com"
!git config user.name "Rickykapoor"

# Set up GitHub auth with Personal Access Token
# Get one at: https://github.com/settings/tokens (tick 'repo' scope)
PAT = ""  # <-- paste your GitHub PAT here
if PAT:
    !git remote set-url origin https://{PAT}@github.com/Rickykapoor/earlyMind.git
    print('GitHub auth configured.')
else:
    print('WARNING: PAT not set. Last cell (git push) will fail. Set PAT above.')

In [ ]:
# CELL 2: Mount Google Drive (for saving checkpoints after training)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/EarlyMind/checkpoints', exist_ok=True)
print('Google Drive mounted. Checkpoints will be saved to MyDrive/EarlyMind/checkpoints/')

In [ ]:
# CELL 3: Install dependencies
!pip install -q mne nibabel nilearn openneuro-py torch torchvision \
  streamlit plotly scikit-learn pytorch-tabnet \
  xgboost catboost tqdm pyyaml scipy

In [ ]:
# CELL 4: Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 5: Download raw MRI dataset from GitHub Releases
!mkdir -p datasets/mri/baby_open_brains
!wget -qO mri_raw.tar.gz https://github.com/Rickykapoor/earlyMind/releases/download/v1.0.0-data/mri_raw.tar.gz
!tar -xzf mri_raw.tar.gz -C datasets/mri
print('Raw MRI dataset ready.')

In [ ]:
import sys, os; sys.path.insert(0, '/content/earlyMind') if '/content/earlyMind' not in sys.path else None; os.chdir('/content/earlyMind') if os.getcwd() != '/content/earlyMind' else None
# CELL 6: Setup paths + inspect participants
import sys
sys.path.insert(0, '/content/earlyMind')

from src.config import cfg
from src.data.mri_loader import process_mri_dataset
import pandas as pd
import numpy as np

print('MRI raw dir: ', cfg.paths.mri_raw)
print('MRI proc dir:', cfg.paths.mri_processed)

tsv_path = cfg.paths.mri_raw / 'participants.tsv'
if tsv_path.exists():
    df_part = pd.read_csv(tsv_path, sep='\t')
    print('Participants TSV columns:', df_part.columns.tolist())
    display(df_part)
else:
    print('participants.tsv not found')

In [ ]:
# CELL 7: Run MRI preprocessing (10 real subjects → datasets/processed/mri/)
results = process_mri_dataset(
    mri_dir=cfg.paths.mri_raw,
    output_dir=cfg.paths.mri_processed,
)

print('\n=== Preprocessing Results ===')
for pid, info in results.items():
    print(f'  {pid}: slices={info["slices"].shape}, age={info["age_months"]:.1f}mo, '
          f'label={info["label"]}, DQ={info["dq"]:.1f}')

import glob
real_files = glob.glob('datasets/processed/mri/*.npz')
print(f'\n  Real .npz files written: {len(real_files)}')

In [ ]:
# CELL 8: Run augmentation (10,000 synthetic samples, seed=42)
# Runtime: ~5-15 min on Colab CPU
print('Running MRI augmentation pipeline...')
!python scripts/balance_mri.py --skip-preprocess --target 10000 --seed 42 --dq-noise-std 5.0

aug_files = glob.glob('datasets/mri/augmented/*.npz')
print(f'\n  Augmented files generated: {len(aug_files):,}')
if len(aug_files) < 9000:
    raise RuntimeError(f'Expected ~10,000 files but got {len(aug_files)}. Check output above.')
print('  Augmentation complete!')

In [ ]:
# CELL 9: Verify class distribution (sample)
import collections
labels = [int(np.load(f)['label']) for f in sorted(aug_files)[:500]]
dist = collections.Counter(labels)
names = {0:'Typical',1:'Borderline',2:'Mild ID',3:'Moderate ID',4:'Severe ID',5:'Profound ID'}
print('=== Augmented Class Distribution (sample of 500) ===')
for k in sorted(dist):
    pct = dist[k]/500*100
    print(f'  {names.get(k,k):12s}: {dist[k]:5d}  ({pct:.1f}%)  {"█"*int(pct/2)}')

In [ ]:
# CELL 10: Save augmented data summary to Google Drive
import shutil
drive_dir = '/content/drive/MyDrive/EarlyMind'
os.makedirs(f'{drive_dir}/reports', exist_ok=True)

# Copy any reports generated
for report in ['mri_slices.png', 'mri_augment_compare.png']:
    src = f'reports/{report}'
    if os.path.exists(src):
        shutil.copy2(src, f'{drive_dir}/reports/{report}')
        print(f'Saved {report} to Drive')

print('Notebook 02 complete. Proceed to notebook 04.')